### Config stuff

In [19]:
import fact
from pyspark.sql.functions import explode

import ConnectionConfig as cc
from delta import DeltaTable
cc.setupEnvironment()

In [20]:
spark = cc.startLocalCluster("factRides")
spark.getActiveSession()

In [21]:
#EXTRACT
cc.set_connectionProfile("velodb")
ride_src_df = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option("dbtable", """
        (SELECT
            r.*,
           haversine_km(
    CAST(SPLIT_PART(TRIM(BOTH '()' FROM CAST(startpoint AS TEXT)), ',', 1) AS NUMERIC),
    CAST(SPLIT_PART(TRIM(BOTH '()' FROM CAST(startpoint AS TEXT)), ',', 2) AS NUMERIC),
    CAST(SPLIT_PART(TRIM(BOTH '()' FROM CAST(endpoint AS TEXT)), ',', 1) AS NUMERIC),
    CAST(SPLIT_PART(TRIM(BOTH '()' FROM CAST(endpoint AS TEXT)), ',', 2) AS NUMERIC)
) AS ride_distance_km
        FROM rides r
        ) AS rides_with_distance
    """) \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()\

ride_src_df.show(20)

+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+--------------------+
|rideid|       startpoint|         endpoint|          starttime|            endtime|vehicleid|subscriptionid|startlockid|endlockid|    ride_distance_km|
+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+--------------------+
|     1|(51.2083,4.44595)|(51.1938,4.40228)|2015-09-22 00:00:00|2012-09-22 00:00:00|      844|         13296|       4849|     3188|3.443440884519500000|
|     2|(51.2174,4.41597)|(51.2188,4.40935)|2015-09-22 00:00:00|2012-09-22 00:00:00|     4545|         45924|       NULL|     NULL|0.486639569093544000|
|     3|(51.2088,4.40834)|(51.2077,4.39846)|2015-09-22 00:00:00|2012-09-22 00:00:00|     3419|         25722|       2046|     1951|0.699051307894188000|
|     4|(51.2023,4.41208)|(51.2119,4.39894)|2015-09-22 00:00:00|2012-09-22 00:00:0

In [22]:
#EXTRACT
cc.set_connectionProfile("velodb")
subscription_df = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option("dbtable", "(select * from subscriptions) as subq") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptionid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()\

subscription_df.show(20)

+--------------+----------+------------------+------+
|subscriptionid| validfrom|subscriptiontypeid|userid|
+--------------+----------+------------------+------+
|             1|2019-08-02|                 3|     1|
|             2|2019-11-12|                 1|     1|
|             3|2020-12-14|                 1|     1|
|             4|2021-10-05|                 2|     2|
|             5|2022-09-17|                 3|     3|
|             6|2019-04-08|                 1|     4|
|             7|2022-05-16|                 3|     5|
|             8|2019-06-29|                 3|     6|
|             9|2023-11-30|                 3|     6|
|            10|2023-12-31|                 3|     6|
|            11|2021-12-01|                 2|     7|
|            12|2019-02-01|                 2|     8|
|            13|2023-11-26|                 3|     8|
|            14|2021-05-08|                 3|     9|
|            15|2023-08-15|                 3|     9|
|            16|2023-11-29| 

In [23]:
ride_src_df.createOrReplaceTempView("rides_source")
subscription_df.createOrReplaceTempView("subscriptions")


#### 2 MAKE DIMENSION TABLES AVAILABLE AS VIEWS

In [24]:
#EXTRACT
dim_date = spark.read.format("delta").load("spark-warehouse/dimdate")
dim_date.createOrReplaceTempView("dimDate")
dim_user = spark.read.format("delta").load("spark-warehouse/dimuser")
dim_user.createOrReplaceTempView("dimUser")
dim_lock = spark.read.format("delta").load("spark-warehouse/dimlock")
dim_lock.createOrReplaceTempView("dimLock")
dim_vehicle = spark.read.format("delta").load("spark-warehouse/dimVehicle")
dim_vehicle.createOrReplaceTempView("dimVehicle")
dim_weather = spark.read.format("delta").load("spark-warehouse/dimWeather")
dim_weather.createOrReplaceTempView("dimWeather")


#### 3 Build the fact table

Within the creation of a fact table always perform these two tasks:
1.   Include the measures of the fact
2.   Use the dimension tables to look up the surrogate keys that correspond with the natural key value. In case of SCD2 dimension use the scd_start en scd_end to find the correct version of the data in the dimension


In [25]:
dim_date.printSchema()
dim_user.printSchema()
ride_src_df.printSchema()
ride_src_df.show(5)

root
 |-- date_SK: long (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month_nr: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- day_nr: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- IsWeekDay: string (nullable = true)

root
 |-- userid: integer (nullable = true)
 |-- street: string (nullable = true)
 |-- number: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- user_SK: string (nullable = true)
 |-- scd_start: timestamp (nullable = true)
 |-- scd_end: timestamp (nullable = true)
 |-- md5: string (nullable = true)
 |-- current: boolean (nullable = true)

root
 |-- rideid: integer (nullable = true)
 |-- startpoint: string (nullable = true)
 |-- endpoint: string (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- endtime: timestamp (nullable

In [26]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import arrays_zip, col, hour, to_date, udf, explode

#EXTRACT
weather_df = spark.read.option("multiLine", True).json("C:/Users/lzkol/PycharmProjects/sparkdelta/weather")

weather_flat = weather_df.withColumn("hourly_row", explode(
    arrays_zip(col("hourly.time"), col("hourly.weathercode"))
)).select(
    col("zipCode"),
    col("date"),
    col("hourly_row.time").alias("weather_time"),
    col("hourly_row.weathercode").alias("weathercode")
)

#EXTRACT
weather_flat = weather_flat.withColumn("weather_hour", hour("weather_time"))
weather_flat = weather_flat.withColumn("weather_date", to_date("weather_time"))

# Map weathercode to type
def weathercode_to_type(code):
    if code in [0, 1]:
        return "Pleasant"
    if code in [2, 3, 45, 48]:
        return "Neutral"
    if code in [51, 53, 55, 61, 63, 65, 80, 81, 82, 95, 96, 99, 71, 73, 75, 77, 85, 86]:
        return "Unpleasant"
    return "Unknown"

weathercode_to_type_udf = udf(weathercode_to_type, StringType())
weather_flat = weather_flat.withColumn("weather_type", weathercode_to_type_udf(col("weathercode")))
weather_flat.printSchema()
weather_flat.show(truncate=False)
weather_flat.createOrReplaceTempView("weather_flat")

root
 |-- zipCode: string (nullable = true)
 |-- date: string (nullable = true)
 |-- weather_time: string (nullable = true)
 |-- weathercode: long (nullable = true)
 |-- weather_hour: integer (nullable = true)
 |-- weather_date: date (nullable = true)
 |-- weather_type: string (nullable = true)

+-------+----------+----------------+-----------+------------+------------+------------+
|zipCode|date      |weather_time    |weathercode|weather_hour|weather_date|weather_type|
+-------+----------+----------------+-----------+------------+------------+------------+
|2100   |2023-06-03|2023-06-03T00:00|0          |0           |2023-06-03  |Pleasant    |
|2100   |2023-06-03|2023-06-03T01:00|0          |1           |2023-06-03  |Pleasant    |
|2100   |2023-06-03|2023-06-03T02:00|0          |2           |2023-06-03  |Pleasant    |
|2100   |2023-06-03|2023-06-03T03:00|0          |3           |2023-06-03  |Pleasant    |
|2100   |2023-06-03|2023-06-03T04:00|0          |4           |2023-06-03  |Pleas

In [27]:
#TRANSFORM
spark.sql("""
CREATE OR REPLACE TEMP VIEW rides AS
SELECT
    r.*,
    s.userid,
    UNIX_TIMESTAMP(r.endtime) - UNIX_TIMESTAMP(r.starttime) AS ride_duration
FROM rides_source r
JOIN subscriptions s ON r.subscriptionid = s.subscriptionid
WHERE
    UNIX_TIMESTAMP(r.endtime) > UNIX_TIMESTAMP(r.starttime)
""")


DataFrame[]

## Initial load
The first time loading the fact table perform a FULL load. All data is written to the Delta Table.
After initial load the code line has to be disabled

In [28]:
#TRANSFORM
rides_fact_df = spark.sql("""
SELECT
    r.rideid,
    d_start.date_sk AS start_date_sk,
    d_end.date_sk   AS end_date_sk,
    l_start.lock_id AS start_lock_id,
    l_end.lock_id   AS end_lock_id,
    u.user_sk       AS user_sk,
    v.vehicle_id    AS vehicle_id,
    r.startpoint,
    r.endpoint,
    r.endtime,
    r.starttime,
    r.ride_duration,
    r.ride_distance_km,
    COALESCE(dw.weather_id, unknown_dw.weather_id) AS weather_id
FROM rides r
LEFT JOIN dimDate d_start ON DATE(r.starttime) = d_start.date
LEFT JOIN dimDate d_end   ON DATE(r.endtime) = d_end.date
LEFT JOIN dimLock l_start ON r.startlockid   = l_start.lock_id
LEFT JOIN dimLock l_end   ON r.endlockid     = l_end.lock_id
LEFT JOIN dimUser u
    ON r.userid = u.userid
    AND r.starttime >= u.scd_start
    AND r.starttime <  u.scd_end
LEFT JOIN dimVehicle v  ON r.vehicleid = v.vehicle_id

LEFT JOIN weather_flat wf
    ON l_start.zipcode = wf.zipCode
    AND DATE(r.starttime) = wf.weather_date
    AND HOUR(r.starttime) = wf.weather_hour

LEFT JOIN dimWeather dw ON wf.weather_type = dw.weather_type
LEFT JOIN dimWeather unknown_dw ON unknown_dw.weather_type = 'Unknown'
""")

from pyspark.sql.functions import md5, concat_ws

cols_to_hash = [
    "rideid", "start_date_sk", "end_date_sk", "start_lock_id", "end_lock_id",
    "user_sk", "vehicle_id", "startpoint", "endpoint", "endtime", "starttime",
    "ride_duration", "ride_distance_km", "weather_id"
]

rides_fact_df = rides_fact_df.withColumn(
    "md5",
    md5(concat_ws("||", *cols_to_hash))
)


+------+-------------+-----------+-------------+-----------+--------------------+----------+-----------------+-----------------+-------------------+-------------------+-------------+--------------------+----------+
|rideid|start_date_sk|end_date_sk|start_lock_id|end_lock_id|             user_sk|vehicle_id|       startpoint|         endpoint|            endtime|          starttime|ride_duration|    ride_distance_km|weather_id|
+------+-------------+-----------+-------------+-----------+--------------------+----------+-----------------+-----------------+-------------------+-------------------+-------------+--------------------+----------+
|    17|         3916|       3916|         2046|       1951|acb69aa3-dd10-4f9...|      3419|(51.2088,4.40834)|(51.2077,4.39846)|2019-09-22 08:30:25|2019-09-22 08:27:38|          167|0.699051307894188000|         4|
|    25|         3916|       3916|          985|       2148|6dee489a-b56e-497...|      1400|(51.2179,4.41777)| (51.211,4.40724)|2019-09-22 0

In [29]:
# Show all rows where weather_id is not 4
# rides_fact_df.filter(rides_fact_df.weather_id == 2).show()
# rides_fact_df.write.format("delta").mode("overwrite").saveAsTable("factRides")

In [30]:
# Current fact table
dt_factRides = DeltaTable.forPath(spark, "./spark-warehouse/factrides")
dt_factRides.toDF().createOrReplaceTempView("factRides_current")

# New fact data
rides_fact_df.createOrReplaceTempView("factRides_new")

#LOAD
result = spark.sql("""
MERGE INTO factRides_current AS target
USING factRides_new AS source
ON target.rideid = source.rideid
WHEN MATCHED AND source.md5 <> target.md5 THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
""")


In [17]:
#DEBUG
spark.sql("SELECT * FROM factRides_current").show()




+------+-------------+-----------+-------------+-----------+--------------------+----------+-----------------+-----------------+-------------------+-------------------+-------------+--------------------+----------+--------------------+
|rideid|start_date_sk|end_date_sk|start_lock_id|end_lock_id|             user_sk|vehicle_id|       startpoint|         endpoint|            endtime|          starttime|ride_duration|    ride_distance_km|weather_id|                 md5|
+------+-------------+-----------+-------------+-----------+--------------------+----------+-----------------+-----------------+-------------------+-------------------+-------------+--------------------+----------+--------------------+
|    22|         3916|       3916|         2572|         13|22082fbb-9b09-47b...|      2015| (51.2191,4.3949)|  (51.2281,4.409)|2019-09-22 08:44:46|2019-09-22 08:39:11|          335|1.402023228415670000|         4|7767a54762da40539...|
|    31|         3916|       3916|          574|       2

### Checking the history of your delta fact table

AttributeError: module 'fact' has no attribute 'history'

In [60]:
spark.stop()